# Induction Heads 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_Mech_Interp/blob/main/exercises/02_induction_heads/notebook_normal.ipynb)

In [ ]:
import os, importlib

if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_Mech_Interp/main/exercises/02_induction_heads/utils.py",
        "utils.py"
    )
    %pip install transformer_lens einops circuitsvis plotly "numpy>=2" "transformers>=4.38,<4.52" -q

import utils
importlib.reload(utils)
from utils import test_average_over_condition, test_head_detectors, test_logit_attribution

In [ ]:
import torch
from torch import Tensor
from typing import Callable
import plotly.express as px
from IPython.display import display
import circuitsvis as cv
from transformer_lens import ActivationCache, HookedTransformer

torch.set_grad_enabled(False)

## Loading Models

We use two models today:

1. **GPT-2 Small** (12 layers, 80M params) — a real model where we can find induction heads
2. **2-layer attention-only model** — a minimal model trained to exhibit induction behavior clearly

The toy model has no MLPs (attention only), making it easier to interpret.

In [ ]:
gpt2 = HookedTransformer.from_pretrained("gpt2-small")
gpt2.set_use_attn_result(True)

model = HookedTransformer.from_pretrained("attn-only-2l")
model.set_use_attn_result(True)

## Caching Activations

`model.run_with_cache(tokens)` returns all intermediate activations.
We can access attention patterns via `cache["pattern", layer]` which has shape `(n_heads, seq_len, seq_len)`.

In [ ]:
text = " I can't repeat enough that ML4Good is the best place to learn about AI safety." * 2
tokens = model.to_tokens(text)
logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)

pattern_layer0 = cache["pattern", 0]
print(f"Attention pattern shape: {pattern_layer0.shape}")
print(f"  = (n_heads={model.cfg.n_heads}, seq_len={tokens.shape[1]}, seq_len={tokens.shape[1]})")

## Visualizing Attention Patterns

We use `circuitsvis` to interactively inspect attention heads.
Hover over a head to see its attention pattern. Click to lock it.

Look for these common patterns:
- **Current-token heads**: attend mostly to the current position (diagonal)
- **Previous-token heads**: attend to the previous position (off-diagonal)
- **First-token heads**: attend heavily to the first token (first column bright)
- **Induction heads**: attend to the token *after* the previous occurrence of the current token

In [ ]:
str_tokens = model.to_str_tokens(text)
for layer in range(model.cfg.n_layers):
    print(f"Layer {layer + 1}")
    attention_pattern = cache["pattern", layer]
    display(cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern))

## Exercise A: Classify Attention Patterns

To systematically detect these head types, we need a function that computes the average attention weight for entries satisfying some condition on `(query_pos, key_pos)`.

For example, to detect current-token heads, we average over entries where `query == key`.

Implement `average_over_condition`: given an attention tensor `(n_heads, n_query, n_key)` and a condition function, return the average attention weight per head over positions where the condition is True.

In [ ]:
def average_over_condition(
    tensor: torch.Tensor,
    condition: Callable[[int, int], bool],
) -> torch.Tensor:
    """
    Args:
        tensor: (n_heads, n_query, n_key) attention pattern
        condition: function(query_idx, key_idx) -> bool
    Returns:
        (n_heads,) average attention where condition is True
    """
    n_heads, n_query, n_key = tensor.shape

    # Step 1: build a boolean mask of shape (n_query, n_key)
    # TODO (~1 line): list comprehension with condition(q, k)
    mask = None

    # Step 2: for each head, average the attention values where mask is True
    # TODO (~1 line): index with mask, take mean
    return None

In [ ]:
test_average_over_condition(average_over_condition)

<details>
<summary><b>Hint</b></summary>

Build a 2D boolean tensor:
`mask = torch.tensor([[condition(q, k) for k in range(n_key)] for q in range(n_query)])`
Then: `torch.stack([tensor[h][mask].mean() for h in range(n_heads)])`
</details>

<details>
<summary><b>Solution</b></summary>

```python
def average_over_condition(tensor, condition):
    n_heads, n_query, n_key = tensor.shape
    mask = torch.tensor([[condition(q, k) for k in range(n_key)] for q in range(n_query)])
    return torch.stack([tensor[h][mask].mean() for h in range(n_heads)])
```
</details>

## Exercise B: Head Type Detectors

Now use `average_over_condition` to build four detector functions.
Each returns a list of head names (e.g. `["L1H3", "L2H7"]`) where the average attention on the relevant pattern exceeds a threshold.

The helpers `heads_above_threshold` and `find_repeating_rows` are provided. You implement the four detector functions:

| Detector | Condition on `(query, key)` |
|---|---|
| `current_attn_detector` | `query == key` |
| `prev_attn_detector` | `query == key + 1` |
| `first_attn_detector` | `key == 0` |
| `induction_attn_detector` | `key` is the position after the previous occurrence of the token at `query` |

In [ ]:
THRESHOLD = 0.35


def heads_above_threshold(cache, condition, threshold):
    result = []
    for layer, pattern in enumerate(cache.stack_activation("pattern")):
        scores = average_over_condition(pattern, condition)
        for head, score in enumerate(scores):
            if score > threshold:
                result.append(f"L{layer+1}H{head}")
    return result


def find_repeating_rows(tokens):
    last_occurrence = {}
    repeats = {}
    for pos, token in enumerate(tokens[0]):
        tid = token.item()
        if tid in last_occurrence:
            repeats[pos] = last_occurrence[tid]
        last_occurrence[tid] = pos
    return repeats


def current_attn_detector(cache, threshold=THRESHOLD):
    # TODO (~2 lines): define condition where query == key, return heads_above_threshold
    pass


def prev_attn_detector(cache, threshold=THRESHOLD):
    # TODO (~2 lines): condition where query == key + 1
    pass


def first_attn_detector(cache, threshold=THRESHOLD):
    # TODO (~2 lines): condition where key == 0
    pass


def induction_attn_detector(cache, tokens, threshold=THRESHOLD):
    # TODO (~3 lines): use find_repeating_rows(tokens), then condition where
    # key == repeat_dict[query] + 1 (token after the previous occurrence)
    pass

In [ ]:
test_head_detectors(current_attn_detector, prev_attn_detector, first_attn_detector, induction_attn_detector, model)

<details>
<summary><b>Hint</b></summary>

Each detector is a one-liner calling `heads_above_threshold` with a lambda.
For induction: `repeat_dict = find_repeating_rows(tokens)`, then condition checks `q in repeat_dict and repeat_dict[q] + 1 == k`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
THRESHOLD = 0.35


def heads_above_threshold(cache, condition, threshold):
    result = []
    for layer, pattern in enumerate(cache.stack_activation("pattern")):
        scores = average_over_condition(pattern, condition)
        for head, score in enumerate(scores):
            if score > threshold:
                result.append(f"L{layer+1}H{head}")
    return result


def find_repeating_rows(tokens):
    last_occurrence = {}
    repeats = {}
    for pos, token in enumerate(tokens[0]):
        tid = token.item()
        if tid in last_occurrence:
            repeats[pos] = last_occurrence[tid]
        last_occurrence[tid] = pos
    return repeats


def current_attn_detector(cache, threshold=THRESHOLD):
    return heads_above_threshold(cache, lambda q, k: q == k, threshold)


def prev_attn_detector(cache, threshold=THRESHOLD):
    return heads_above_threshold(cache, lambda q, k: q == k + 1, threshold)


def first_attn_detector(cache, threshold=THRESHOLD):
    return heads_above_threshold(cache, lambda q, k: k == 0, threshold)


def induction_attn_detector(cache, tokens, threshold=THRESHOLD):
    repeat_dict = find_repeating_rows(tokens)
    return heads_above_threshold(cache, lambda q, k: q in repeat_dict and repeat_dict[q] + 1 == k, threshold)
```
</details>

### Detection Results

In [ ]:
print("Current-token heads:", current_attn_detector(cache))
print("Previous-token heads:", prev_attn_detector(cache))
print("First-token heads:", first_attn_detector(cache))
print("Induction heads:", induction_attn_detector(cache, tokens))

### Induction Heads in GPT-2

In [ ]:
gpt2_tokens = gpt2.to_tokens(text)
_, gpt2_cache = gpt2.run_with_cache(gpt2_tokens, remove_batch_dim=True)

print("Induction heads in GPT-2:")
print(induction_attn_detector(gpt2_cache, gpt2_tokens, threshold=0.2))

# Logit Attribution

We now implement a second method to identify induction heads: **logit attribution**.

Because of the residual stream, the output logits are the sum of contributions from each attention head.
We can decompose the logits to see which heads contribute most to predicting the correct next token.

For a given token position $t$:
1. Get the output of each attention head at position $t$ from `cache["blocks.{layer}.attn.hook_result"]` (shape `(seq_len, n_heads, d_model)`)
2. Unembed each head's output to get its contribution to the vocabulary logits
3. Look at the logit for the *correct* next token `tokens[0, t+1]`

Heads with high attribution at repeated positions are likely induction heads.

## Exercise C: Logit Attribution

In [ ]:
def plot_attribution(scores):
    n_layers, n_heads = scores.shape
    fig = px.imshow(
        scores.detach().cpu().numpy(),
        labels=dict(x="Head", y="Layer"),
        x=[f"H{i}" for i in range(n_heads)],
        y=[f"L{i+1}" for i in range(n_layers)],
        color_continuous_scale="RdBu_r",
        color_continuous_midpoint=0,
    )
    fig.update_layout(title="Logit Attribution per Head", width=600, height=400)
    fig.show()

In [ ]:
def logit_attribution(tokens, model, cache, token_position):
    """
    Args:
        tokens: (1, seq_len) token IDs
        model: HookedTransformer
        cache: ActivationCache from run_with_cache
        token_position: position to compute attribution for
    Returns:
        (n_layers, n_heads) attribution scores for the correct next token
    """
    # Step 1: collect attention head outputs for each layer
    # cache[f"blocks.{i}.attn.hook_result"] has shape (seq_len, n_heads, d_model)
    # TODO (~1 line): list of tensors, one per layer
    results = None

    # Step 2: stack into (seq_len, n_layers, n_heads, d_model) and select token_position
    # TODO (~2 lines)
    results = None

    # Step 3: unembed to get (n_layers, n_heads, vocab_size)
    # TODO (~1 line)
    logits = None

    # Step 4: get the logit for the correct next token
    # TODO (~1 line): index with tokens[0, token_position + 1]
    return None

In [ ]:
test_logit_attribution(logit_attribution, model)

<details>
<summary><b>Hint</b></summary>

Stack with `torch.stack(results, dim=1)`, then index `[token_position]` to get `(n_layers, n_heads, d_model)`.
`model.unembed` handles the rest. The correct next token is `tokens[0, token_position + 1]`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def logit_attribution(tokens, model, cache, token_position):
    results = [cache[f"blocks.{i}.attn.hook_result"] for i in range(model.cfg.n_layers)]
    results = torch.stack(results, dim=1)
    results = results[token_position]
    logits = model.unembed(results)
    return logits[:, :, tokens[0, token_position + 1]]
```
</details>

### Comparing Methods

In [ ]:
repeat_tokens = model.to_tokens(text)
_, toy_cache = model.run_with_cache(repeat_tokens, remove_batch_dim=True)
_, gpt2_cache2 = gpt2.run_with_cache(gpt2.to_tokens(text), remove_batch_dim=True)

token_position = 14
print(f"Attribution at position {token_position} (toy model):")
scores = logit_attribution(repeat_tokens, model, toy_cache, token_position)
plot_attribution(scores)

print(f"\nAttribution at position {token_position} (GPT-2):")
gpt2_tokens = gpt2.to_tokens(text)
scores = logit_attribution(gpt2_tokens, gpt2, gpt2_cache2, token_position)
plot_attribution(scores)

<details>
<summary><b>What to look for</b></summary>

- In the toy model, Layer 2 heads should show strong positive attribution (these are the induction heads).
- In GPT-2, induction heads tend to cluster in layers 5-6.
- The two methods (attention patterns and logit attribution) should broadly agree on which heads are induction heads.
</details>